# Modeling Exam (Hard) — CoderPad Practice Problems

Harder problems with more interacting components. 30 min each. All formulas provided. Do MEDIUM first.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math
from typing import Any, Optional

---

## Problem 1: Adaptive Layer Norm + DiT Block

### Scenario

DiT (Diffusion Transformer) conditions each layer on a timestep embedding by modulating LayerNorm parameters. Build the full DiT block from scratch.

### Formulas

**Standard LayerNorm:** `LN(x) = (x - mean) / sqrt(var + eps)`  (no learned affine)

**Adaptive LayerNorm (adaLN):** Predict scale/shift from conditioning signal:
```
gamma, beta = Linear(cond).chunk(2, dim=-1)   # cond_dim → 2 * dim
adaLN(x, cond) = (1 + gamma) * LN(x) + beta   # (1 + gamma) is a residual scale
```
Broadcast `cond` (B, cond_dim) across the sequence dimension T.

**Multi-Head Self-Attention:**
```
Q, K, V = x @ W_q, x @ W_k, x @ W_v      # (B, T, dim) → (B, T, dim)
Reshape to (B, num_heads, T, head_dim)
attn = softmax(Q @ K^T / sqrt(head_dim)) @ V
out = reshape(attn) @ W_o                   # back to (B, T, dim)
```

**DiT Block:**
```
x = x + attn(adaLN_1(x, cond))              # pre-norm attention + residual
x = x + mlp(adaLN_2(x, cond))               # pre-norm MLP + residual
```
MLP: Linear(dim, dim*4) → GELU → Linear(dim*4, dim)

### Requirements

**`AdaptiveLayerNorm(nn.Module)`:**
- `__init__(self, dim: int, cond_dim: int)`
- Uses `nn.LayerNorm(dim, elementwise_affine=False)` internally
- Linear projection: cond_dim → 2 * dim (for gamma and beta)
- `forward(self, x: Tensor, cond: Tensor)` — x: (B, T, dim), cond: (B, cond_dim) → (B, T, dim)

**`DiTBlock(nn.Module)`:**
- `__init__(self, dim: int, cond_dim: int, num_heads: int)`
- Two separate AdaptiveLayerNorm instances (one for attention, one for MLP)
- Multi-head self-attention (Q/K/V projections + output projection)
- MLP with expansion ratio 4
- `forward(self, x: Tensor, cond: Tensor)` — x: (B, T, dim), cond: (B, cond_dim) → (B, T, dim)

In [ ]:
class AdaptiveLayerNorm(nn.Module):
    # YOUR CODE HERE
    pass


class DiTBlock(nn.Module):
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 1 ---
torch.manual_seed(42)

# AdaptiveLayerNorm basics
aln = AdaptiveLayerNorm(dim=64, cond_dim=32)
x = torch.randn(2, 10, 64)
cond = torch.randn(2, 32)
out_aln = aln(x, cond)
assert out_aln.shape == (2, 10, 64), f"adaLN shape: expected (2,10,64), got {out_aln.shape}"

# Different conditions produce different outputs
cond2 = torch.randn(2, 32)
out_aln2 = aln(x, cond2)
assert not torch.allclose(out_aln, out_aln2, atol=1e-3), "Different cond should give different output"

# DiTBlock shape
dit = DiTBlock(dim=64, cond_dim=32, num_heads=4)
x_dit = torch.randn(2, 10, 64)
cond_dit = torch.randn(2, 32)
out_dit = dit(x_dit, cond_dit)
assert out_dit.shape == (2, 10, 64), f"DiT shape: expected (2,10,64), got {out_dit.shape}"

# Conditioning affects output
cond_dit2 = torch.randn(2, 32)
out_dit2 = dit(x_dit, cond_dit2)
assert not torch.allclose(out_dit, out_dit2, atol=1e-3), "Different cond should change DiT output"

# Residual connection: output should not be wildly different from input at init
diff = (out_dit - x_dit).abs().mean()
input_scale = x_dit.abs().mean()
assert diff < input_scale * 10, f"Residual block should not explode at init (diff={diff:.3f}, input={input_scale:.3f})"

# Gradient flows through all parameters
loss = out_dit.sum()
loss.backward()
for name, p in dit.named_parameters():
    assert p.grad is not None, f"No gradient for {name}"
    assert not torch.isnan(p.grad).any(), f"NaN gradient for {name}"

# num_heads must divide dim
dit_8h = DiTBlock(dim=64, cond_dim=32, num_heads=8)
out_8h = dit_8h(x_dit, cond_dit)
assert out_8h.shape == (2, 10, 64), "Should work with 8 heads"

# Batch size 1
out_b1 = dit(x_dit[:1], cond_dit[:1])
assert out_b1.shape == (1, 10, 64), "Should work with batch size 1"

print("Problem 1: ALL TESTS PASSED")

---

## Problem 2: LoRA (Low-Rank Adaptation)

### Scenario

Fine-tuning large models is expensive. LoRA adds trainable low-rank matrices to frozen weights, reducing trainable parameters by 100x+.

### Formulas

**Standard linear:** `y = x @ W^T + b` where W is (out_features, in_features)

**LoRA forward:**
```
y = x @ W^T + b + (x @ A^T @ B^T) * scale
```
- A: (rank, in_features) — initialized from N(0, 1)
- B: (out_features, rank) — initialized to zeros
- scale: alpha / rank (a constant)
- W and b are **frozen** (require_grad=False)
- Only A and B are trainable

**Merge:** After training, fold LoRA into the base weight:
```
W_merged = W + (B @ A) * scale
```
Result is a single nn.Linear with no LoRA overhead.

### Requirements

**`LoRALinear(nn.Module)`:**
- `__init__(self, base_linear: nn.Linear, rank: int, alpha: float = 1.0)`
- Freeze base_linear weights and bias
- Create trainable A (rank, in_features) and B (out_features, rank)
- A initialized with `nn.init.kaiming_uniform_`, B initialized to zeros
- `forward(self, x)` — compute original + low-rank delta
- `merge(self) -> nn.Linear` — return a new nn.Linear with merged weights

**`inject_lora(model: nn.Module, rank: int, alpha: float = 1.0) -> nn.Module`:**
- Replace ALL nn.Linear layers in the model with LoRALinear wrappers
- Handle nested modules (use named_modules + setattr)
- Return the modified model

**`merge_lora(model: nn.Module) -> nn.Module`:**
- Replace ALL LoRALinear layers with their merged nn.Linear equivalents
- Return the modified model

In [ ]:
class LoRALinear(nn.Module):
    # YOUR CODE HERE
    pass


def inject_lora(model: nn.Module, rank: int, alpha: float = 1.0) -> nn.Module:
    # YOUR CODE HERE
    pass


def merge_lora(model: nn.Module) -> nn.Module:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 2 ---
torch.manual_seed(42)

# Basic LoRALinear
base = nn.Linear(64, 32)
lora = LoRALinear(base, rank=4, alpha=1.0)
x = torch.randn(2, 64)
out = lora(x)
assert out.shape == (2, 32), f"Expected (2,32), got {out.shape}"

# B initialized to zeros → LoRA delta is zero at init → output matches base
base_out = base(x)
assert torch.allclose(out, base_out, atol=1e-5), "At init (B=0), LoRA output should match base"

# Base weights are frozen
for p in [lora.base_linear.weight, lora.base_linear.bias]:
    assert not p.requires_grad, "Base weights should be frozen"

# LoRA weights are trainable
assert lora.A.requires_grad, "A should be trainable"
assert lora.B.requires_grad, "B should be trainable"

# Trainable param count is much smaller
total_base = sum(p.numel() for p in base.parameters())
trainable_lora = sum(p.numel() for p in lora.parameters() if p.requires_grad)
assert trainable_lora < total_base, f"LoRA params ({trainable_lora}) should be < base ({total_base})"

# Training changes LoRA output
optimizer = torch.optim.SGD([p for p in lora.parameters() if p.requires_grad], lr=0.1)
target = torch.randn(2, 32)
loss = F.mse_loss(lora(x), target)
loss.backward()
optimizer.step()
out_after = lora(x)
assert not torch.allclose(out_after, base_out, atol=1e-5), "After training, LoRA output should differ"

# Merge produces equivalent nn.Linear
merged = lora.merge()
assert isinstance(merged, nn.Linear), "merge() should return nn.Linear"
merged_out = merged(x)
lora_out = lora(x)
assert torch.allclose(merged_out, lora_out, atol=1e-5), "Merged output should match LoRA output"

# inject_lora replaces all Linear layers
model = nn.Sequential(
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Linear(128, 32),
)
model_lora = inject_lora(model, rank=4)
num_lora = sum(1 for m in model_lora.modules() if isinstance(m, LoRALinear))
assert num_lora == 2, f"Expected 2 LoRALinear, got {num_lora}"
num_plain_linear = sum(1 for m in model_lora.modules() if type(m) == nn.Linear)
# The base_linear inside LoRALinear are nn.Linear but not direct children of Sequential
# The Sequential should have LoRALinear, not plain nn.Linear
assert isinstance(model_lora[0], LoRALinear), "First layer should be LoRALinear"
assert isinstance(model_lora[2], LoRALinear), "Third layer should be LoRALinear"

# merge_lora restores plain nn.Linear
x_test = torch.randn(3, 64)
out_before_merge = model_lora(x_test)
model_merged = merge_lora(model_lora)
num_lora_after = sum(1 for m in model_merged.modules() if isinstance(m, LoRALinear))
assert num_lora_after == 0, "After merge, no LoRALinear should remain"
out_after_merge = model_merged(x_test)
assert torch.allclose(out_before_merge, out_after_merge, atol=1e-5), "Merged model should give same output"

print("Problem 2: ALL TESTS PASSED")

---

## Problem 3: DDPM Reverse Process (Sampling)

### Scenario

Given a trained noise predictor and a noise schedule, implement the full DDPM reverse sampling loop to generate images from pure noise.

### Formulas

**Noise schedule (given):**
```
beta_t ∈ (0, 1)               — noise added at step t
alpha_t = 1 - beta_t
alpha_bar_t = prod(alpha_1 ... alpha_t)   — cumulative product
```

**Reverse step (t → t-1):**
```
mu_t = (1 / sqrt(alpha_t)) * (x_t - (beta_t / sqrt(1 - alpha_bar_t)) * eps_theta(x_t, t))
sigma_t = sqrt(beta_t)
x_{t-1} = mu_t + sigma_t * z     where z ~ N(0, I) if t > 1, else z = 0
```

### Requirements

**`DDPMSchedule`:**
- `__init__(self, betas: torch.Tensor)` — betas is (T,) tensor
- Precompute and store: `alphas`, `alpha_bars`, `sqrt_alpha`, `sqrt_one_minus_alpha_bar`

**`ddpm_sample(model_fn, schedule: DDPMSchedule, shape: tuple, device: str = "cpu") -> torch.Tensor`:**
- `model_fn(x_t, t)` predicts noise. x_t is (B, C, H, W), t is (B,) integer timesteps.
- Start from x_T ~ N(0, I)
- Iterate from t=T-1 down to t=0
- At each step, apply the reverse formula
- At t=0, do NOT add noise (z=0)
- Return final x_0 with shape `shape`

In [ ]:
class DDPMSchedule:
    # YOUR CODE HERE
    pass


def ddpm_sample(
    model_fn,
    schedule: DDPMSchedule,
    shape: tuple,
    device: str = "cpu",
) -> torch.Tensor:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 3 ---
torch.manual_seed(42)

T = 20
betas = torch.linspace(1e-4, 0.02, T)
schedule = DDPMSchedule(betas)

# Schedule precomputation checks
assert schedule.alphas.shape == (T,)
assert schedule.alpha_bars.shape == (T,)
assert torch.allclose(schedule.alphas, 1.0 - betas)
assert torch.allclose(schedule.alpha_bars, torch.cumprod(schedule.alphas, dim=0))
assert (schedule.alpha_bars > 0).all(), "alpha_bars should be positive"
assert schedule.alpha_bars[-1] < schedule.alpha_bars[0], "alpha_bar should decrease"

# Dummy noise predictor: just returns zeros (predicts no noise)
def dummy_model(x_t, t):
    return torch.zeros_like(x_t)

shape = (2, 3, 8, 8)
torch.manual_seed(42)
samples = ddpm_sample(dummy_model, schedule, shape)
assert samples.shape == shape, f"Expected {shape}, got {samples.shape}"
assert samples.dtype == torch.float32

# Output should differ from pure noise (the reverse process transforms it)
torch.manual_seed(42)
pure_noise = torch.randn(shape)
assert not torch.allclose(samples, pure_noise, atol=1e-3), "Output should differ from initial noise"

# Deterministic with same seed
torch.manual_seed(42)
samples2 = ddpm_sample(dummy_model, schedule, shape)
assert torch.allclose(samples, samples2), "Same seed should give same result"

# Different seed gives different result
torch.manual_seed(99)
samples3 = ddpm_sample(dummy_model, schedule, shape)
assert not torch.allclose(samples, samples3), "Different seed should give different result"

# Model that predicts the actual noise should recover ~zero mean
# (with enough steps and a perfect model, the mean should be near 0)
def identity_model(x_t, t):
    return x_t  # not a perfect noise predictor, but tests that model output is used

torch.manual_seed(42)
samples_id = ddpm_sample(identity_model, schedule, shape)
assert samples_id.shape == shape
# With identity model, output should be different from dummy model
assert not torch.allclose(samples, samples_id, atol=1e-2), "Different models should give different results"

print("Problem 3: ALL TESTS PASSED")

---

## Problem 4: Classifier-Free Guidance

### Scenario

Classifier-free guidance (CFG) improves conditional generation quality by amplifying the difference between conditioned and unconditioned predictions.

### Formulas

**CFG output:**
```
eps_cfg = eps_uncond + guidance_scale * (eps_cond - eps_uncond)
```
Equivalently:
```
eps_cfg = (1 - guidance_scale) * eps_uncond + guidance_scale * eps_cond
```

- `guidance_scale = 1.0` → pure conditioned output
- `guidance_scale = 0.0` → pure unconditioned output  
- `guidance_scale > 1.0` → amplified conditioning (typical: 3.0-15.0)

**Training:** Randomly drop conditioning with probability `p_uncond` (replace with null/zero embedding).

### Requirements

**`CFGWrapper(nn.Module)`:**
- `__init__(self, model: nn.Module, guidance_scale: float = 7.5)`
- The wrapped `model` has signature: `model(x, t, cond)` → predicted noise
  - When `cond` is None, model runs unconditionally
- `forward(self, x, t, cond)` — runs the model twice:
  1. Unconditional: `model(x, t, None)` → eps_uncond
  2. Conditional: `model(x, t, cond)` → eps_cond
  3. Combine with CFG formula
  4. Return combined output

**`cfg_sample_step(cfg_wrapper, x_t, t_idx, schedule, guidance_scale, cond) -> torch.Tensor`:**
- Perform one DDPM reverse step using CFG-guided noise prediction
- Temporarily set the wrapper's guidance_scale, compute guided prediction, apply reverse formula
- Same reverse formula as Problem 3
- Return x_{t-1}

In [ ]:
class CFGWrapper(nn.Module):
    # YOUR CODE HERE
    pass


def cfg_sample_step(
    cfg_wrapper: CFGWrapper,
    x_t: torch.Tensor,
    t_idx: int,
    schedule: DDPMSchedule,
    guidance_scale: float,
    cond: torch.Tensor,
) -> torch.Tensor:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 4 ---
torch.manual_seed(42)

# Simple model: returns different noise depending on conditioning
class SimpleCondModel(nn.Module):
    def forward(self, x, t, cond=None):
        if cond is None:
            return x * 0.1  # unconditioned: small noise
        else:
            return x * 0.1 + cond.unsqueeze(-1).unsqueeze(-1) * 0.5  # conditioned: adds cond signal

model = SimpleCondModel()
x = torch.randn(2, 3, 8, 8)
t = torch.tensor([5, 5])
cond = torch.randn(2, 3)

# CFGWrapper basics
cfg = CFGWrapper(model, guidance_scale=7.5)
out_cfg = cfg(x, t, cond)
assert out_cfg.shape == x.shape, f"CFG output shape should match input: {out_cfg.shape}"

# guidance_scale=1.0 should give pure conditioned output
cfg_1 = CFGWrapper(model, guidance_scale=1.0)
out_scale1 = cfg_1(x, t, cond)
out_cond = model(x, t, cond)
assert torch.allclose(out_scale1, out_cond, atol=1e-5), "scale=1 should give conditioned output"

# guidance_scale=0.0 should give pure unconditioned output
cfg_0 = CFGWrapper(model, guidance_scale=0.0)
out_scale0 = cfg_0(x, t, cond)
out_uncond = model(x, t, None)
assert torch.allclose(out_scale0, out_uncond, atol=1e-5), "scale=0 should give unconditioned output"

# guidance_scale > 1 should amplify difference
cfg_high = CFGWrapper(model, guidance_scale=10.0)
out_high = cfg_high(x, t, cond)
diff_high = (out_high - out_uncond).abs().mean()
diff_cond = (out_cond - out_uncond).abs().mean()
assert diff_high > diff_cond, "Higher guidance should amplify conditioning"

# cfg_sample_step: uses schedule from Problem 3
# (Requires DDPMSchedule to be implemented)
T = 10
betas_cfg = torch.linspace(1e-4, 0.02, T)
sched_cfg = DDPMSchedule(betas_cfg)

torch.manual_seed(42)
x_t = torch.randn(2, 3, 8, 8)
x_prev = cfg_sample_step(cfg, x_t, t_idx=5, schedule=sched_cfg, guidance_scale=7.5, cond=cond)
assert x_prev.shape == x_t.shape, "Output shape should match input"
assert not torch.allclose(x_prev, x_t, atol=1e-3), "Reverse step should change x"

# Different guidance scales give different results
torch.manual_seed(42)
x_prev_low = cfg_sample_step(cfg, x_t, t_idx=5, schedule=sched_cfg, guidance_scale=1.0, cond=cond)
torch.manual_seed(42)
x_prev_high = cfg_sample_step(cfg, x_t, t_idx=5, schedule=sched_cfg, guidance_scale=15.0, cond=cond)
assert not torch.allclose(x_prev_low, x_prev_high, atol=1e-3), "Different scales should give different steps"

print("Problem 4: ALL TESTS PASSED")

---

## Problem 5: Knowledge Distillation Loss

### Scenario

Knowledge distillation trains a smaller student model to mimic a larger teacher by matching soft probability distributions.

### Formulas

**Soft labels (temperature scaling):**
```
p_teacher = softmax(teacher_logits / T)
p_student = softmax(student_logits / T)
```

**KL divergence (soft loss):**
```
KL(p_teacher || p_student) = sum(p_teacher * log(p_teacher / p_student))
```
Use `F.kl_div` with `log_softmax` for numerical stability:
```
soft_loss = F.kl_div(log_softmax(student/T), softmax(teacher/T), reduction='batchmean')
```

**Hard loss (standard CE):**
```
hard_loss = F.cross_entropy(student_logits, true_labels)
```

**Total distillation loss:**
```
loss = alpha * hard_loss + (1 - alpha) * T^2 * soft_loss
```
The T^2 factor corrects for the gradient magnitude change from temperature scaling.

### Requirements

**`DistillationLoss(nn.Module)`:**
- `__init__(self, temperature: float = 4.0, alpha: float = 0.5)`
- `forward(self, student_logits, teacher_logits, true_labels) -> dict`
  - student_logits: (B, C), teacher_logits: (B, C), true_labels: (B,) integers
  - Return `{"total_loss": Tensor, "hard_loss": Tensor, "soft_loss": Tensor}`
  - teacher_logits should be detached (no gradient through teacher)

**`DistillationTrainer`:**
- `__init__(self, teacher: nn.Module, student: nn.Module, temperature: float = 4.0, alpha: float = 0.5)`
- `train_step(self, x: Tensor, labels: Tensor) -> dict` — run both models, compute loss, return loss dict
  - Teacher runs in `torch.no_grad()` mode
  - Only student parameters should have gradients

In [ ]:
class DistillationLoss(nn.Module):
    # YOUR CODE HERE
    pass


class DistillationTrainer:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 5 ---
torch.manual_seed(42)

# Basic DistillationLoss
kd_loss = DistillationLoss(temperature=4.0, alpha=0.5)
student_logits = torch.randn(8, 10, requires_grad=True)
teacher_logits = torch.randn(8, 10)
labels = torch.randint(0, 10, (8,))

result = kd_loss(student_logits, teacher_logits, labels)
assert "total_loss" in result and "hard_loss" in result and "soft_loss" in result
assert result["total_loss"].ndim == 0, "Loss should be scalar"
assert result["hard_loss"].ndim == 0
assert result["soft_loss"].ndim == 0

# Total loss = alpha * hard + (1-alpha) * T^2 * soft
expected_total = 0.5 * result["hard_loss"] + 0.5 * 16.0 * result["soft_loss"]
assert torch.allclose(result["total_loss"], expected_total, atol=1e-5), \
    f"Total loss formula mismatch: {result['total_loss'].item():.4f} vs {expected_total.item():.4f}"

# Gradient flows through student, not teacher
result["total_loss"].backward()
assert student_logits.grad is not None, "Student should have gradients"
assert not torch.isnan(student_logits.grad).any()

# When student matches teacher perfectly, soft loss should be ~0
same_logits = torch.randn(8, 10)
result_same = kd_loss(same_logits, same_logits.detach().clone(), labels)
assert result_same["soft_loss"].item() < 1e-5, f"Soft loss should be ~0 when student=teacher, got {result_same['soft_loss'].item()}"

# Loss should be non-negative
assert result["total_loss"].item() >= 0
assert result["soft_loss"].item() >= 0

# DistillationTrainer
teacher = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 10))
student = nn.Sequential(nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 10))  # smaller
trainer = DistillationTrainer(teacher, student, temperature=4.0, alpha=0.5)

x = torch.randn(4, 32)
y = torch.randint(0, 10, (4,))
step_result = trainer.train_step(x, y)
assert step_result["total_loss"].ndim == 0

# Teacher should have no gradients
for p in teacher.parameters():
    assert p.grad is None or (p.grad == 0).all(), "Teacher should not receive gradients"

# Student should have gradients
has_grad = any(p.grad is not None and p.grad.abs().sum() > 0 for p in student.parameters())
assert has_grad, "Student should have gradients after train_step"

print("Problem 5: ALL TESTS PASSED")

---

## Problem 6: Multi-Scale Feature Pyramid

### Scenario

Feature Pyramid Networks (FPN) combine multi-scale features from a backbone to produce feature maps with consistent channel dimensions at all scales. Used in detection, segmentation, and diffusion U-Nets.

### Architecture

**Backbone** produces features at 3 scales (given, you don't build this):
```
C3: (B, ch3, H/8, W/8)      — deepest, most channels, smallest spatial
C2: (B, ch2, H/4, W/4)      — mid
C1: (B, ch1, H/2, W/2)      — shallowest, fewest channels, largest spatial
```

**FPN (top-down path):**
```
P3 = Conv1x1(C3, out_channels)                                    # lateral
P2 = Conv3x3(Conv1x1(C2, out_channels) + Upsample2x(P3))         # lateral + top-down
P1 = Conv3x3(Conv1x1(C1, out_channels) + Upsample2x(P2))         # lateral + top-down
```

- `Conv1x1`: lateral connection (channel projection, no padding)
- `Upsample2x`: nearest-neighbor 2x upsampling
- `Conv3x3`: smoothing conv with padding=1 to preserve spatial dims
- All outputs P1, P2, P3 have `out_channels` channels

### Requirements

**`FeaturePyramid(nn.Module)`:**
- `__init__(self, in_channels: list[int], out_channels: int)` — in_channels is [ch1, ch2, ch3] (shallowest to deepest)
- 3 lateral Conv1x1 layers
- 2 smoothing Conv3x3 layers (for P1 and P2; P3 doesn't need smoothing)
- `forward(self, features: list[torch.Tensor]) -> list[torch.Tensor]`
  - features = [C1, C2, C3] (shallowest to deepest)
  - Returns [P1, P2, P3] (shallowest to deepest, all with out_channels)

In [ ]:
class FeaturePyramid(nn.Module):
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 6 ---
torch.manual_seed(42)

in_channels = [64, 128, 256]  # C1, C2, C3
out_channels = 128
B = 2

fpn = FeaturePyramid(in_channels, out_channels)

# Create multi-scale features
C1 = torch.randn(B, 64, 32, 32)    # shallowest, largest spatial
C2 = torch.randn(B, 128, 16, 16)   # mid
C3 = torch.randn(B, 256, 8, 8)     # deepest, smallest spatial

P1, P2, P3 = fpn([C1, C2, C3])

# All outputs have out_channels
assert P1.shape == (B, out_channels, 32, 32), f"P1 shape: expected {(B, out_channels, 32, 32)}, got {P1.shape}"
assert P2.shape == (B, out_channels, 16, 16), f"P2 shape: expected {(B, out_channels, 16, 16)}, got {P2.shape}"
assert P3.shape == (B, out_channels, 8, 8), f"P3 shape: expected {(B, out_channels, 8, 8)}, got {P3.shape}"

# Spatial dimensions preserved at each level
assert P1.shape[2:] == C1.shape[2:], "P1 spatial dims should match C1"
assert P2.shape[2:] == C2.shape[2:], "P2 spatial dims should match C2"
assert P3.shape[2:] == C3.shape[2:], "P3 spatial dims should match C3"

# Gradient flow through all paths
loss = sum(p.sum() for p in [P1, P2, P3])
loss.backward()
for name, p in fpn.named_parameters():
    assert p.grad is not None, f"No gradient for {name}"
    assert not torch.isnan(p.grad).any(), f"NaN gradient in {name}"

# Top-down path: P1 should depend on C3 (information flows from deep to shallow)
C3_modified = C3.clone()
C3_modified += 10.0  # big change to deepest features
P1_mod, P2_mod, P3_mod = fpn([C1, C2, C3_modified])
assert not torch.allclose(P1, P1_mod, atol=1e-3), "P1 should be affected by changes in C3 (top-down path)"
assert not torch.allclose(P2, P2_mod, atol=1e-3), "P2 should be affected by changes in C3"

# Different in_channels configuration
fpn_small = FeaturePyramid([32, 64, 128], out_channels=64)
C1s = torch.randn(1, 32, 16, 16)
C2s = torch.randn(1, 64, 8, 8)
C3s = torch.randn(1, 128, 4, 4)
P1s, P2s, P3s = fpn_small([C1s, C2s, C3s])
assert P1s.shape[1] == 64 and P2s.shape[1] == 64 and P3s.shape[1] == 64

print("Problem 6: ALL TESTS PASSED")

---

## Problem 7: Exponential Moving Average (EMA) Model

### Scenario

EMA maintains a smoothed copy of model weights during training. The EMA model typically produces better/more stable outputs than the raw training model. Used in nearly all diffusion model training.

### Formula

**EMA update (after each training step):**
```
shadow_param = decay * shadow_param + (1 - decay) * model_param
```
Typical decay: 0.999 or 0.9999 (EMA weights change slowly).

### Requirements

**`EMAModel`:**
- `__init__(self, model: nn.Module, decay: float = 0.999)`
- Creates a deep copy of model parameters as shadow parameters
- Does NOT create a second model — just stores the parameter tensors (use `state_dict` / manual copy)
- `update(self, model: nn.Module)` — one EMA update step for all parameters
- `get_ema_model(self, model: nn.Module) -> nn.Module` — returns a copy of model with shadow weights loaded
- `store(self, model: nn.Module)` — save current model params (for later restore)
- `restore(self, model: nn.Module)` — restore model params from last `store()` call
- EMA should work with `torch.no_grad()` (no gradient computation during updates)

In [ ]:
class EMAModel:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 7 ---
torch.manual_seed(42)

# Simple model
model = nn.Sequential(nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 8))
ema = EMAModel(model, decay=0.99)

# At init, EMA params should match model params
ema_model = ema.get_ema_model(model)
x = torch.randn(4, 16)
assert torch.allclose(model(x), ema_model(x), atol=1e-6), "EMA should match model at init"

# Simulate training: update model weights, then update EMA
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
for _ in range(10):
    loss = model(x).sum()
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    ema.update(model)

# After training, EMA params should differ from model params
ema_model_after = ema.get_ema_model(model)
model_out = model(x)
ema_out = ema_model_after(x)
assert not torch.allclose(model_out, ema_out, atol=1e-3), "EMA should diverge from model during training"

# EMA params should be between initial and current (smoothed)
# Since decay=0.99, EMA moves slowly → EMA output magnitude should be between
# We can't check exact values easily, but we can verify EMA is not identical to model
ema_state = ema_model_after.state_dict()
model_state = model.state_dict()
for key in model_state:
    assert not torch.allclose(ema_state[key], model_state[key], atol=1e-4), \
        f"EMA param {key} should differ from model after training"

# store/restore
ema.store(model)
original_out = model(x).clone()

# Modify model weights
with torch.no_grad():
    for p in model.parameters():
        p.add_(torch.randn_like(p) * 10)  # big perturbation
perturbed_out = model(x)
assert not torch.allclose(original_out, perturbed_out, atol=1e-2), "Perturbation should change output"

# Restore should bring back original weights
ema.restore(model)
restored_out = model(x)
assert torch.allclose(original_out, restored_out, atol=1e-6), "Restore should recover original weights"

# EMA update should not create gradients
model.zero_grad()
ema.update(model)
for p in model.parameters():
    assert p.grad is None or (p.grad == 0).all(), "EMA update should not create gradients"

# High decay (0.9999): EMA barely moves
model2 = nn.Linear(8, 4)
ema_slow = EMAModel(model2, decay=0.9999)
initial_state = {k: v.clone() for k, v in model2.state_dict().items()}
with torch.no_grad():
    model2.weight += 1.0  # shift weights
ema_slow.update(model2)
ema_slow_model = ema_slow.get_ema_model(model2)
# EMA should have barely moved from initial
for key in initial_state:
    diff = (ema_slow_model.state_dict()[key] - initial_state[key]).abs().max().item()
    assert diff < 0.01, f"High-decay EMA should barely move: {key} moved by {diff}"

print("Problem 7: ALL TESTS PASSED")

---

## Problem 8: Noise-Conditioned Score Network

### Scenario

Score matching learns the gradient of the log data density (the "score"). A noise-conditioned score network takes noisy data and the noise level as input, and predicts the score at that noise level.

### Formulas

**Sinusoidal embedding for noise level σ:**
```
freqs = exp(-ln(10000) * i / (d/2))    for i = 0, ..., d/2 - 1
embed = [sin(σ * freqs), cos(σ * freqs)]   # concatenate → dim d
```

**Noisy data:**
```
x_noisy = x_clean + sigma * noise    where noise ~ N(0, I)
```

**True score at noise level σ (for Gaussian perturbation):**
```
score(x_noisy, σ) = -(x_noisy - x_clean) / σ^2 = -noise / σ
```
(Since x_noisy - x_clean = σ * noise, so score = -σ*noise/σ^2 = -noise/σ)

**Denoising Score Matching loss:**
```
loss = E_σ[ σ^2 * E_{x, noise}[ ||score_net(x_noisy, σ) - (-noise/σ)||^2 ] ]
```
The σ^2 weighting ensures all noise levels contribute equally.

### Requirements

**`SinusoidalEmbedding(nn.Module)`:**
- `__init__(self, dim: int)` — embedding dimension
- `forward(self, sigma: Tensor)` — sigma: (B,) → (B, dim)

**`ScoreNetwork(nn.Module)`:**
- `__init__(self, data_dim: int, hidden_dim: int = 128, embed_dim: int = 64)`
- MLP architecture: concat(x, sigma_embed) → Linear → SiLU → Linear → SiLU → Linear → output
- Input: x (B, data_dim) + sigma embedding (B, embed_dim) = (B, data_dim + embed_dim)
- Output: (B, data_dim) — predicted score, same dim as input data
- `forward(self, x, sigma)` — x: (B, data_dim), sigma: (B,) → (B, data_dim)

**`dsm_loss(score_net, x_clean, sigmas) -> torch.Tensor`:**
- x_clean: (B, data_dim), sigmas: (B,) noise levels
- Sample noise ~ N(0, I)
- Compute x_noisy, target score, predicted score
- Return weighted MSE loss (scalar)

In [ ]:
class SinusoidalEmbedding(nn.Module):
    # YOUR CODE HERE
    pass


class ScoreNetwork(nn.Module):
    # YOUR CODE HERE
    pass


def dsm_loss(
    score_net: ScoreNetwork,
    x_clean: torch.Tensor,
    sigmas: torch.Tensor,
) -> torch.Tensor:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 8 ---
torch.manual_seed(42)

# SinusoidalEmbedding
embed = SinusoidalEmbedding(dim=64)
sigmas = torch.tensor([0.01, 0.1, 1.0, 10.0])
sigma_emb = embed(sigmas)
assert sigma_emb.shape == (4, 64), f"Embedding shape: expected (4,64), got {sigma_emb.shape}"

# Different sigmas should produce different embeddings
assert not torch.allclose(sigma_emb[0], sigma_emb[1], atol=1e-3), "Different sigmas → different embeddings"
assert not torch.allclose(sigma_emb[1], sigma_emb[2], atol=1e-3)

# Same sigma should produce same embedding
sigma_same = torch.tensor([0.5, 0.5])
emb_same = embed(sigma_same)
assert torch.allclose(emb_same[0], emb_same[1]), "Same sigma should give same embedding"

# ScoreNetwork
data_dim = 16
score_net = ScoreNetwork(data_dim=data_dim, hidden_dim=128, embed_dim=64)
x = torch.randn(8, data_dim)
sigma = torch.ones(8) * 0.1
score_pred = score_net(x, sigma)
assert score_pred.shape == (8, data_dim), f"Score shape: expected (8,{data_dim}), got {score_pred.shape}"

# Score network output depends on sigma
sigma2 = torch.ones(8) * 1.0
score_pred2 = score_net(x, sigma2)
assert not torch.allclose(score_pred, score_pred2, atol=1e-3), "Different sigma should give different scores"

# dsm_loss
torch.manual_seed(42)
x_clean = torch.randn(16, data_dim)
sigmas_batch = torch.rand(16) * 0.5 + 0.01  # random sigmas in [0.01, 0.51]
loss = dsm_loss(score_net, x_clean, sigmas_batch)
assert loss.ndim == 0, f"Loss should be scalar, got ndim={loss.ndim}"
assert loss.item() > 0, "Loss should be positive"
assert not torch.isnan(loss), "Loss should not be NaN"

# Gradient flows
loss.backward()
has_grad = any(p.grad is not None and p.grad.abs().sum() > 0 for p in score_net.parameters())
assert has_grad, "Gradient should flow through score network"

# Training should reduce loss (basic sanity)
score_net_train = ScoreNetwork(data_dim=4, hidden_dim=64, embed_dim=32)
optimizer = torch.optim.Adam(score_net_train.parameters(), lr=1e-3)
x_train = torch.randn(32, 4)  # fixed data
sigmas_train = torch.ones(32) * 0.1  # fixed sigma

torch.manual_seed(42)
loss_before = dsm_loss(score_net_train, x_train, sigmas_train).item()
for _ in range(100):
    optimizer.zero_grad()
    torch.manual_seed(42)  # same noise each step for stable training
    loss = dsm_loss(score_net_train, x_train, sigmas_train)
    loss.backward()
    optimizer.step()
loss_after = loss.item()
assert loss_after < loss_before, f"Training should reduce loss: {loss_before:.4f} → {loss_after:.4f}"

print("Problem 8: ALL TESTS PASSED")

---

## Scoring

| Result | Assessment |
|--------|------------|
| Solved in < 25 min, all tests pass | **Strong pass** |
| Solved in 25-30 min, minor debugging | **Pass** — ready for harder problems |
| 30-40 min or needed hints | **Borderline** — redo tomorrow |
| Couldn't solve or > 40 min | **Go back to MEDIUM** — nail those first |